In this file, we will work on the classification task that will help predict the Visit Mode of any particular user using various inputs.

In [1]:
# Importing the libraries
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Loading the dataset
df_clean = pd.read_csv('../clean_data/df_clean.csv')

# Checking few sample rows of the dataset
df_clean.sample(7)

,TransactionId,UserId,VisitYear,VisitMonth,VisitModeId,AttractionId,Rating,CityId,CityName,CountryId,...,RegionId,Region,ContinentId,Continent,AttractionCityId,AttractionTypeId,Attraction,AttractionType,VisitMode,Month
33592,69935,74287,2018,4,2,748,5,2263,Ulaanbaatar,66,...,10,Central Asia,3,Asia,1,72,Tegalalang Rice Terrace,Points of Interest & Landmarks,Couples,April
38097,100240,72941,2018,1,2,749,4,3164,Medan,101,...,14,South East Asia,3,Asia,1,93,Tegenungan Waterfall,Waterfalls,Couples,January
21506,34019,63923,2016,7,3,673,2,7153,Puerto de la Cruz,157,...,20,Southern Europe,5,Europe,1,13,Seminyak Beach,Beaches,Family,July
24085,37971,82946,2015,11,2,481,4,1676,Phoenix,51,...,8,Northern America,2,America,1,13,Nusa Dua Beach,Beaches,Couples,November
16606,25209,65903,2016,8,3,841,5,3687,Adelaide,109,...,15,Australia,4,Australia & Oceania,1,92,Waterbom Bali,Water Parks,Family,August
22181,34907,28533,2014,8,2,673,4,4940,Perth,109,...,15,Australia,4,Australia & Oceania,1,13,Seminyak Beach,Beaches,Couples,August
1638,2748,55312,2017,4,2,640,5,426,London,48,...,8,Northern America,2,America,1,63,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,Couples,April


In [3]:
# Checking the name of the columns
df_clean.columns

Index(['TransactionId', 'UserId', 'VisitYear', 'VisitMonth', 'VisitModeId',
       'AttractionId', 'Rating', 'CityId', 'CityName', 'CountryId', 'Country',
       'RegionId', 'Region', 'ContinentId', 'Continent', 'AttractionCityId',
       'AttractionTypeId', 'Attraction', 'AttractionType', 'VisitMode',
       'Month'],
      dtype='object')

In [ ]:
# Creating a new dataframe with only the columns we need for the classification task
df_clf = df_clean[['Month', 'CityName', 'Country', 'Region', 'Continent', 'Attraction', 'AttractionType', 'Rating', 'VisitMode']].copy()

# Checking the rows of this dataframe
df_clf.sample(7)

,Month,CityName,Country,Region,Continent,Attraction,AttractionType,Rating,VisitMode
875,June,Miami,United States,Northern America,America,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5,Couples
11457,January,Melbourne,United States,Northern America,America,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,4,Couples
24902,March,New York City,United States,Northern America,America,Sanur Beach,Beaches,5,Couples
23328,July,Melbourne,United States,Northern America,America,Nusa Dua Beach,Beaches,5,Couples
13695,September,Whitianga,New Zealand,Oceania,Australia & Oceania,Waterbom Bali,Water Parks,5,Family
43426,January,Spalding,United Kingdom,Western Europe,Europe,Kuta Beach - Bali,Beaches,3,Couples
3864,January,Manama,Bahrain,Middle East,Asia,Sacred Monkey Forest Sanctuary,Nature & Wildlife Areas,5,Couples


In [5]:
# Train test split
from sklearn.model_selection import train_test_split

# Separating the features and the target variable
x = df_clf.drop('VisitMode', axis=1)
y = df_clf['VisitMode']

# Splitting the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42, stratify=y)

In [6]:
# Creating a list of the categorical columns
cat_cols = ['Month', 'CityName', 'Country', 'Region', 'Continent', 'Attraction', 'AttractionType']

# Creating cat_features to be used later in the CatBoostClassifier model
cat_features = [x_train.columns.get_loc(col) for col in cat_cols]

In [7]:
# For LightGBM we will convert the categorical columns to category dtype
for c in cat_cols:
    x_train[c] = x_train[c].astype('category')
    x_test[c] = x_test[c].astype('category')

In [9]:
# Importing the necessary libraries
import mlflow
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

In [10]:
# We  will be using class_weight to handle the class imbalance in the target variable
# This increases the weight of the minority classes
from sklearn.utils.class_weight import compute_class_weight
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

- CatBoost Models' MLFlow run and training.

In [11]:
# Setting the MLflow experiment name
mlflow.set_experiment('Tourism_VisitMode_Classification')

# Starting the MLflow run
with mlflow.start_run(run_name='CatBoostClassifier_1'):

    # Initializing the CatBoostClassifier model
    cat1 = CatBoostClassifier(iterations=300,
                              depth=6,
                              learning_rate=0.1,
                              loss_function='MultiClass',
                              class_weights=class_weights,
                              random_seed=42)
    
    # Fitting the model on the training data
    cat1.fit(x_train, y_train, cat_features=cat_features, eval_set=(x_test, y_test), verbose=1)
    
    # Predicting the Visit Mode on the test data
    y_pred = cat1.predict(x_test)
    
    # Evaluating the model performance
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'CatBoostClassifier')
    mlflow.log_param('iterations', 300)
    mlflow.log_param('depth', 6)
    mlflow.log_param('learning_rate', 0.1)
    mlflow.log_metric('f1_score', f1)
    mlflow.log_metric('recall_score', recall)
    mlflow.log_metric('precision_score', precision)

    # Logging the model to MLflow
    mlflow.catboost.log_model(cat1, 'cat1')

2026/02/21 20:20:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/21 20:20:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/21 20:20:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/21 20:20:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/21 20:20:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/21 20:20:48 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/21 20:20:50 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/21 20:20:50 INFO alembic.runtime.migration: Will assume non-transactional DDL.


0:	learn: 1.5852302	test: 1.5868203	best: 1.5868203 (0)	total: 1.36s	remaining: 6m 47s
1:	learn: 1.5687132	test: 1.5710907	best: 1.5710907 (1)	total: 2.66s	remaining: 6m 36s
2:	learn: 1.5546658	test: 1.5579001	best: 1.5579001 (2)	total: 3.6s	remaining: 5m 56s
3:	learn: 1.5429744	test: 1.5490818	best: 1.5490818 (3)	total: 4.6s	remaining: 5m 40s
4:	learn: 1.5307486	test: 1.5366444	best: 1.5366444 (4)	total: 5.33s	remaining: 5m 14s
5:	learn: 1.5205717	test: 1.5281319	best: 1.5281319 (5)	total: 6s	remaining: 4m 53s
6:	learn: 1.5100603	test: 1.5193075	best: 1.5193075 (6)	total: 6.7s	remaining: 4m 40s
7:	learn: 1.5010809	test: 1.5125871	best: 1.5125871 (7)	total: 7.35s	remaining: 4m 28s
8:	learn: 1.4937494	test: 1.5072987	best: 1.5072987 (8)	total: 7.97s	remaining: 4m 17s
9:	learn: 1.4886236	test: 1.5037879	best: 1.5037879 (9)	total: 8.75s	remaining: 4m 13s
10:	learn: 1.4831570	test: 1.5008707	best: 1.5008707 (10)	total: 9.51s	remaining: 4m 9s
11:	learn: 1.4775263	test: 1.4966192	best: 1.496

2026/02/21 20:25:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


- Cat1 model scores-

    - f1-score: 0.4472.
    - recall score: 0.4245.
    - precision score: 0.4972.

In [12]:
# Starting the MLflow run
with mlflow.start_run(run_name='CatBoostClassifier_2'):

    # Initializing the CatBoostClassifier model
    cat2 = CatBoostClassifier(iterations=800,
                              depth=8,
                              learning_rate=0.01,
                              loss_function='MultiClass',
                              class_weights=class_weights,
                              random_seed=42)
    
    # Fitting the model on the training data
    cat2.fit(x_train, y_train, cat_features=cat_features, eval_set=(x_test, y_test), verbose=1)
    
    # Predicting the Visit Mode on the test data
    y_pred = cat2.predict(x_test)
    
    # Evaluating the model performance
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'CatBoostClassifier')
    mlflow.log_param('iterations', 800)
    mlflow.log_param('depth', 6)
    mlflow.log_param('learning_rate', 0.3)
    mlflow.log_metric('f1_score', f1)
    mlflow.log_metric('recall_score', recall)
    mlflow.log_metric('precision_score', precision)

    # Logging the model to MLflow
    mlflow.catboost.log_model(cat2, 'cat2')

0:	learn: 1.6064711	test: 1.6068985	best: 1.6068985 (0)	total: 3.61s	remaining: 48m 4s
1:	learn: 1.6034189	test: 1.6042454	best: 1.6042454 (1)	total: 6.55s	remaining: 43m 35s
2:	learn: 1.6007122	test: 1.6017850	best: 1.6017850 (2)	total: 9.25s	remaining: 40m 57s
3:	learn: 1.5982033	test: 1.5996053	best: 1.5996053 (3)	total: 11.6s	remaining: 38m 32s
4:	learn: 1.5955774	test: 1.5972107	best: 1.5972107 (4)	total: 14.2s	remaining: 37m 33s
5:	learn: 1.5932098	test: 1.5950552	best: 1.5950552 (5)	total: 16.7s	remaining: 36m 55s
6:	learn: 1.5909700	test: 1.5930364	best: 1.5930364 (6)	total: 19s	remaining: 35m 49s
7:	learn: 1.5885698	test: 1.5909615	best: 1.5909615 (7)	total: 21.1s	remaining: 34m 51s
8:	learn: 1.5859398	test: 1.5888094	best: 1.5888094 (8)	total: 23.1s	remaining: 33m 53s
9:	learn: 1.5835356	test: 1.5866947	best: 1.5866947 (9)	total: 25.2s	remaining: 33m 8s
10:	learn: 1.5816495	test: 1.5848845	best: 1.5848845 (10)	total: 27.5s	remaining: 32m 53s
11:	learn: 1.5792603	test: 1.58290

2026/02/21 20:55:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


- Cat2 model scores-

    - f1-score: 0.4446.
    - recall score: 0.4250.
    - precision score: 0.5017.

- LightGBM Models' MLFlow run and training.

In [13]:
# Importing the libraries
from lightgbm import LGBMClassifier

In [14]:
# Starting the MLflow run
with mlflow.start_run(run_name='LGBMClassifier_1'):
    
    # Initializing the LGBMClassifier model
    light1 = LGBMClassifier(n_estimators=300,
                          max_depth=6,
                          learning_rate=0.1,
                          random_state=42,
                          class_weight=class_weights)
    
    # Fitting the model on the training data
    light1.fit(x_train, y_train, eval_set=[(x_test, y_test)], categorical_feature=cat_cols)
    
    # Predicting the ratings on the test data
    y_pred = light1.predict(x_test)
    
    # Evaluating the model performance
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'LGBMClassifier')
    mlflow.log_param('n_estimators', 300)
    mlflow.log_param('max_depth', 6)
    mlflow.log_param('learning_rate', 0.1)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('precision', precision)

    # Logging the model to MLflow
    mlflow.lightgbm.log_model(light1, 'light1')

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005876 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1793
[LightGBM] [Info] Number of data points in the train set: 37023, number of used features: 8
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

2026/02/21 20:57:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 20:57:15 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.


- Light1 model scores-
    
    - f1-score: 0.4301.
    - recall score: 0.4190.
    - precision score: 0.4487.

In [15]:
# Starting abither MLflow run with different parameters
with mlflow.start_run(run_name='LGBMClassifier_2'):
    
    # Initializing the LGBMClassifier model
    light2 = LGBMClassifier(n_estimators=800,
                          max_depth=8,
                          learning_rate=0.01,
                          random_state=42,
                          class_weight=class_weights)
    
    # Fitting the model on the training data
    light2.fit(x_train, y_train, eval_set=[(x_test, y_test)], categorical_feature=cat_cols)
    
    # Predicting the ratings on the test data
    y_pred = light2.predict(x_test)
    
    # Evaluating the model performance
    f1 = f1_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    precision = precision_score(y_test, y_pred, average='weighted')
    
    # Logging the parameters and metrics to MLflow
    mlflow.log_param('model_name', 'LGBMClassifier')
    mlflow.log_param('n_estimators', 800)
    mlflow.log_param('max_depth', 8)
    mlflow.log_param('learning_rate', 0.01)
    mlflow.log_metric('f1', f1)
    mlflow.log_metric('recall', recall)
    mlflow.log_metric('precision', precision)

    # Logging the model to MLflow
    mlflow.lightgbm.log_model(light2, 'light2')

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006766 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1793
[LightGBM] [Info] Number of data points in the train set: 37023, number of used features: 8
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

2026/02/21 20:59:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/21 20:59:04 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is the 'skops' format.


- Light2 model scores-

    - f1-score: 0.4178.
    - recall score: 0.4022
    - precision score: 0.4469.

- From the MLFlow dashboard observations, it was concluded that the "cat2" model of CatBoostClassifier was the best performing model.

    We will now train this model on the full dataset of df_clf, and then save it for it to be used in the Streamlit app.

In [8]:
# Separating the features and the target variable for classification task
x = df_clf.drop('VisitMode', axis=1)
y = df_clf['VisitMode']

In [17]:
# Initializing the CatBoostClassifier model
cat_clf_final = CatBoostClassifier(iterations=800,
                              depth=8,
                              learning_rate=0.01,
                              loss_function='MultiClass',
                              class_weights=class_weights,
                              random_seed=42)

# Fitting the model on the training data
cat_clf_final.fit(x, y, cat_features=cat_features)

0:	learn: 1.6064218	total: 2.92s	remaining: 38m 54s
1:	learn: 1.6036152	total: 5.66s	remaining: 37m 37s
2:	learn: 1.6007800	total: 7.55s	remaining: 33m 24s
3:	learn: 1.5983649	total: 10.1s	remaining: 33m 28s
4:	learn: 1.5958658	total: 12.2s	remaining: 32m 19s
5:	learn: 1.5931141	total: 14.3s	remaining: 31m 32s
6:	learn: 1.5911580	total: 15s	remaining: 28m 21s
7:	learn: 1.5889102	total: 17s	remaining: 28m 7s
8:	learn: 1.5865801	total: 19.3s	remaining: 28m 17s
9:	learn: 1.5842227	total: 21.2s	remaining: 27m 51s
10:	learn: 1.5816429	total: 23.5s	remaining: 28m 3s
11:	learn: 1.5794124	total: 25.5s	remaining: 27m 51s
12:	learn: 1.5769951	total: 27.6s	remaining: 27m 51s
13:	learn: 1.5749774	total: 29.6s	remaining: 27m 41s
14:	learn: 1.5729025	total: 31.7s	remaining: 27m 37s
15:	learn: 1.5707146	total: 34s	remaining: 27m 47s
16:	learn: 1.5686483	total: 36.2s	remaining: 27m 45s
17:	learn: 1.5668433	total: 38.7s	remaining: 27m 59s
18:	learn: 1.5647717	total: 40.8s	remaining: 27m 57s
19:	learn: 

In [18]:
# Saving the trained final model
cat_clf_final.save_model('../models/cat_clf_final.cbm')

In [10]:
import joblib

# Storing the name of the columns of df_clf
clf_feature_cols = x.columns.tolist()

# Saving the feature columns
joblib.dump(clf_feature_cols, '../artifacts/clf_feature_cols.pkl')

# Saving cat_cols
joblib.dump(cat_cols, '../artifacts/clf_cat_cols.pkl')

['../artifacts/clf_cat_cols.pkl']